# Matrix Operations
**Topic:** Linear Algebra for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Apply** matrix addition, subtraction, and scalar multiplication element by element
- **Explain** why two matrices must have the same shape to be added together
- **Interpret** how adding a bias vector to a weight matrix updates every example in a batch simultaneously

> **Tip:** Start by selecting **Add** from the operation dropdown and hover over a cell in the result matrix. The tooltip shows which cells from A and B combined to produce it. Switch to **Scalar multiply** and notice that every cell is scaled by the same factor.

---
## How we got here

In *01: Scalars, Vectors & Matrices* we defined what matrices are. Now we ask: what can you do with them? The first operations are the ones that mirror familiar arithmetic, addition, subtraction, and multiplication by a number. These simple operations turn out to have direct ML interpretations that appear in every training step.

---
## Why this matters for data science

Adding a bias vector to the output of a neural network layer is matrix addition. Scaling all weights by a constant (learning rate × gradient update) is scalar multiplication applied to a matrix. When a mini-batch of 32 data points passes through a layer, every one of them gets its bias added simultaneously through the same matrix addition, no loop required. Understanding these operations makes training code readable and bug-free.

---
## Try it yourself

In [2]:
out = Output()

_A = np.array([[1.0, 2.0, 3.0],
               [4.0, 5.0, 6.0],
               [7.0, 8.0, 9.0]])
_B = np.array([[9.0, 8.0, 7.0],
               [6.0, 5.0, 4.0],
               [3.0, 2.0, 1.0]])

op_dd = Dropdown(
    options=["Add  (A + B)", "Subtract  (A − B)", "Scalar multiply  (α × A)"],
    value="Add  (A + B)",
    description="Operation:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"))

alpha_sl = FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1,
    description="Scalar α =",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"))

def render(change=None):
    op = op_dd.value
    alpha = alpha_sl.value

    if "Add" in op:
        result = _A + _B
        op_label = "C = A + B  (elementwise addition)"
        fig = make_subplots(rows=1, cols=3,
            subplot_titles=["A", "B", op_label])
        for col, (M, cs) in enumerate([(_A, "Blues"), (_B, "Greens"), (result, "Oranges")], 1):
            fig.add_trace(go.Heatmap(z=M, colorscale=cs, showscale=False,
                text=M.round(2), texttemplate="%{text}"), row=1, col=col)
    elif "Subtract" in op:
        result = _A - _B
        op_label = "C = A − B  (elementwise subtraction)"
        fig = make_subplots(rows=1, cols=3,
            subplot_titles=["A", "B", op_label])
        for col, (M, cs) in enumerate([(_A, "Blues"), (_B, "Greens"), (result, "RdBu")], 1):
            kw = dict(zmid=0) if col == 3 else {}
            fig.add_trace(go.Heatmap(z=M, colorscale=cs, showscale=False,
                text=M.round(2), texttemplate="%{text}", **kw), row=1, col=col)
    else:
        result = alpha * _A
        op_label = f"C = {alpha:.1f} × A  (scalar multiplication)"
        fig = make_subplots(rows=1, cols=2,
            subplot_titles=["A", op_label])
        for col, (M, cs) in enumerate([(_A, "Blues"), (result, "Oranges")], 1):
            fig.add_trace(go.Heatmap(z=M, colorscale=cs, showscale=False,
                text=M.round(2), texttemplate="%{text}"), row=1, col=col)

    layout = base_layout(title=op_label, xaxis_title="", yaxis_title="")
    layout.update(height=310, showlegend=False)
    fig.update_layout(layout)
    fig.update_yaxes(autorange="reversed")

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

op_dd.observe(render, names="value")
alpha_sl.observe(render, names="value")
display(VBox([HBox([op_dd, alpha_sl]), out]))
render()

---
## What's happening?

Matrix arithmetic mirrors number arithmetic but applies element by element across the entire grid at once.

**Matrix addition:** Add corresponding elements. Both matrices must have the same shape.

$$\mathbf{C} = \mathbf{A} + \mathbf{B} \quad \Rightarrow \quad C_{ij} = A_{ij} + B_{ij}$$

**Scalar multiplication:** Multiply every element by the same number.

$$\mathbf{C} = \alpha \mathbf{A} \quad \Rightarrow \quad C_{ij} = \alpha \cdot A_{ij}$$

| Operation | Shape requirement | What it does | ML use case |
|-----------|-----------------|-------------|------------|
| A + B | A and B must be same shape | Add corresponding elements | Adding bias to a weight matrix |
| A − B | A and B must be same shape | Subtract corresponding elements | Computing the gradient update: W − α·∇W |
| α · A | A any shape, α a scalar | Scale every element | Applying learning rate: α × gradient |
| Broadcasting A + b | b is a vector; A is a matrix | Add b to every row of A | Adding bias to every example in a batch |

### Broadcasting: the important extension

NumPy extends addition to allow a vector to be added to every row of a matrix without writing a loop. When a neural network layer adds a bias vector **b** of shape (k,) to an output matrix of shape (n, k), NumPy broadcasts **b** across all n rows automatically. This is not new math, it is just efficient notation for "add this vector to every row."

```python
Z = X @ W + b   # X is (n, p), W is (p, k), b is (k,)
                 # b broadcasts across all n rows of (X @ W)
```

Return to the widget and try adding a shape (2,3) matrix to a shape (3,2) matrix, observe the error, then fix it by selecting Scalar multiply instead.

---
## Real-world example: The weight update step in gradient descent

Every gradient descent step applies three matrix operations in sequence: compute the gradient (a matrix), scale it by the learning rate (scalar multiplication), and subtract it from the current weights (matrix subtraction). The chart shows these three operations acting on a small weight matrix from a neural network layer.

Notice:

- **Notice:** The gradient matrix has the same shape as the weight matrix, it must, because each element of the gradient updates the corresponding weight
- **Notice:** Scaling by the learning rate (0.01) makes all the gradient entries small; without this scaling the update would be too large and overshoot the minimum
- **Notice:** The updated weight matrix differs from the original by a small amount in each cell, this is one gradient descent step, and after thousands of such steps the weights converge to values that minimize the loss

> **Discussion question:** If the learning rate is 0.01 and the gradient for one weight is 15.3, by how much does that weight change in one step? If you saw the gradient jump to 1500 suddenly, what might have gone wrong in the pipeline that computed it?

In [3]:
np.random.seed(42)

W        = np.random.randn(3, 4) * 0.5
gradient = np.random.randn(3, 4) * 2.0
lr       = 0.01
W_new    = W - lr * gradient

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["W", "lr × ∇W", "W − lr × ∇W"])

for col, (M, cs) in enumerate([(W, "Blues"), (lr*gradient, "RdBu"), (W_new, "RdBu")], 1):
    fig.add_trace(go.Heatmap(
        z=M, colorscale=cs, zmid=0, showscale=False,
        text=M.round(3), texttemplate="%{text}",
    ), row=1, col=col)

fig.update_layout(
    title=dict(
        text="Gradient Descent Step: subtract the scaled gradient from each weight",
        font=dict(size=FONT["size_title"])),
    paper_bgcolor=PALETTE["background"],
    height=340,
    margin=dict(t=80, b=20, l=20, r=20))
fig.update_yaxes(autorange="reversed")
display(go.FigureWidget(fig))

FigureWidget({
    'data': [{'colorscale': [[0.0, 'rgb(247,251,255)'], [0.125,
                             'rgb(222,235,247)'], [0.25, 'rgb(198,219,239)'],
                             [0.375, 'rgb(158,202,225)'], [0.5,
                             'rgb(107,174,214)'], [0.625, 'rgb(66,146,198)'],
                             [0.75, 'rgb(33,113,181)'], [0.875, 'rgb(8,81,156)'],
                             [1.0, 'rgb(8,48,107)']],
              'showscale': False,
              'text': {'bdata': ('WDm0yHa+zz9Ei2zn+6mxvyPb+X5qvN' ... 'JNYhBY0T8ZBFYOLbLNv23n+6nx0s2/'),
                       'dtype': 'f8',
                       'shape': '3, 4'},
              'texttemplate': '%{text}',
              'type': 'heatmap',
              'uid': '5c7ef685-1953-4f05-a4b0-6cbe6bd5a191',
              'xaxis': 'x',
              'yaxis': 'y',
              'z': {'bdata': ('fDCpKCrKzz8qBd4FpbKxv2heJFDdud' ... '1q4aZc0T9wG8GuoqjNv+fvFiuEzs2/'),
                    'dtype': 'f8',
                   

### Matrix operation rules and ML interpretations

| Operation | Rule | Shape requirement | ML use case |
|-----------|------|-----------------|-------------|
| A + B | Elementwise addition | Same shape | Add bias; accumulate gradients |
| A − B | Elementwise subtraction | Same shape | Gradient update: W ← W − α·∇W |
| α · A | Scale every element | α is scalar | Apply learning rate to gradient |
| A + b (broadcast) | Add b to every row of A | b.shape == (A.cols,) | Add bias vector to every batch example |
| Elementwise A * B | Hadamard product | Same shape | Dropout masks, attention weights |

---
## Key takeaway

> **Matrix addition and scalar multiplication work element by element and require matching shapes; together with broadcasting they power every gradient descent update, bias addition, and batch normalization step in neural network training.**

---
*Next up: The Dot Product — the single most important operation in all of machine learning, and the building block that matrix multiplication is made from*